# Cài đặt các Thư viện sử dụng

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from underthesea import word_tokenize
import os

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Cài đặt hiển thị dataFrame

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Đọc dữ liệu


- vn_news_223_tdlfr.csv là file chứa news cần xử lý
- vietnamese-stopwords.txt chứa các stop word cần xử lý

In [ ]:

file = os.path.join("../../Data_Train", "Vietnamese.csv")
data_df = pd.read_csv(file, encoding = 'utf-8')

file_stopword = os.path.join("../../Data_Train", "vietnamese-stopwords-dash.txt")
with open(file_stopword, 'r', encoding='utf-8') as file:
    stopwords = file.read().split('\n')

# Khám phá dữ liệu

## Mẫu dữ liệu

In [ ]:
data_df.sample(1)

## Xem thông tin

In [ ]:
data_df.info()

## Xem mô tả

In [ ]:
data_df.describe().round(1)

## Xem số dòng và số cột của dữ liệu

In [ ]:
num_rows=data_df.shape[0]
num_cols=data_df.shape[1]
print(num_rows)
print(num_cols)

## ý nghĩa của các dòng

### Kiểm tra dữ liệu có bị lặp ?

In [ ]:
dup=data_df.index.duplicated().sum()
dup

### ý nghĩa các cột

- text: nội dung của tin tức
- domain: đường dẫn đến trang web chứa tin tức 
- label: nhãn phân biệt tin giả hay tin thật

#### Cột có dtype là object nghĩa là sao?

- Trong Pandas, kiểu dữ liệu object thường ám chỉ chuỗi, nhưng thật ra kiểu dữ liệu object có thể chứa một đối tượng bất kỳ trong Python (vì thật ra ở bên dưới kiểu dữ liệu object chứa địa chỉ).
- Nếu một cột trong dataframe có dtype là object thì có thể các phần tử trong cột này sẽ có kiểu dữ liệu khác nhau
- Để biết được kiểu dữ liệu thật sự của các phần tử trong cột này thì ta phải truy xuất vào từng phần tử. Ta muốn xem thử trong nội bộ mỗi cột này có các kiểu dữ liệu nào

In [ ]:
def open_object_dtype(s):
    dtypes = set()
    s=s.apply(type)
    dtypes.update(s.unique().tolist())
    return dtypes

In [ ]:
open_object_dtype(data_df['title'])

In [ ]:
open_object_dtype(data_df['text'])

In [ ]:
open_object_dtype(data_df['label'])

### Dữ liệu có bị thiếu không?

In [ ]:
data_df.isnull().sum()

#### Kiểm tra phân bố các class có chênh lệch không?

In [ ]:
# data quantity chart
def quantity_chart():
    counts_df1 = data_df['label'].value_counts()

    plt.figure(figsize=(6, 6))

    plt.pie(counts_df1, labels=counts_df1, autopct='%1.1f%%', startangle=90)
    plt.title('Bảng phân phối tin thật và tin giả')

    labels = ['Real' if label == 0 else 'Fake' for label in counts_df1.index]
    plt.legend(labels=labels, loc="best", fontsize=18)  # Tăng kích thước nhãn trong chú thích
    plt.tight_layout()
    plt.show()
quantity_chart()

#### Các thông tin thống kê

##### Chiều dài trung bình mỗi record là bao nhiêu?

In [ ]:
len_sum=0
for i in data_df['text']:
    len_sum+=len(i)
len_avg=len_sum/data_df['text'].count()
len_avg

##### Record dài nhất là bao nhiêu?

In [ ]:
max_len=0
for i in data_df['text']:
    if len(i)>max_len:
        max_len=len(i)
max_len

##### Record ngắn nhất là bao nhiêu?

In [ ]:
min_len=len(data_df['text'][0])
for i in data_df['text']:
    if len(i)<min_len:
        min_len=len(i)
min_len

### Tiền xử lí văn bản tiếng việt

#### Loại bỏ các đường link và các dấu câu, lowercase

In [ ]:
from pyvi import ViTokenizer
from collections import Counter

In [ ]:
def wordopt(text):
    text = text.lower()
    text = re.sub('https?:\/\/.*[\r\n]*', ' ', text)
    text = re.sub('[^\w\s]', ' ', text) 
    text = re.sub('\n', ' ', text)
    return text

#delete numbers
def delete_numbers(text):
    return re.sub(r'\d+', ' ', text)
#lower case
def lower(text):
    return text.lower()

#delete special characters
def remove_special_characters(text):
    ## Remove punctuations
    text = re.sub('[%s]' % re.escape("""!–"#$%&'()*+,،-./:;<=>؟?@[\]^`{|}~“”…"""), ' ', text)
    text = text.replace('؛',"", )
    text = re.sub('\s+', ' ', text)
    text =  " ".join(text.split())
    return text.strip()
#delete stop words
def remove_stopwords(text):
    clean_tokens = ' '.join([word for word in text.split() if word not in stopwords])
    return clean_tokens

def preprocess_nostop(text):
    text = delete_numbers(text)
    text = lower(text)
    text = remove_special_characters(text)
    return text


In [ ]:
# Compound Vietnamese word
def tokenizerVN(text):
    return ViTokenizer.tokenize(text)
# Tokenizer
def tokenizer(text):
    return word_tokenize(text)
# Count token
def count_token(text):
    word = tokenizerVN(str(text))
    return len(word.split())
# Word Cloud
def top_count(dt):
    top = Counter([item for sublist in data_df[dt].apply(lambda x:str(x).split()) for item in sublist])
    temp = pd.DataFrame(top.most_common(50))
    temp.columns = ['Common_words','count']
    temp.style.background_gradient(cmap='Blues')
    return temp

In [ ]:
data_df['article'] = ' |title| '+ data_df["title"] +' |text| '+  data_df["text"] 

In [ ]:
data_df.head(1)

In [ ]:
data_df['Compound_Content'] = data_df['article'].apply(tokenizerVN)

In [ ]:
data_df['Compound_Content_SW'] = data_df['Compound_Content'].apply(preprocess_nostop)

In [ ]:
data_df.head(1)

# Vector hóa

#### Tokenizer

In [ ]:
def tokenize(sentence):
    return tokenizerVN(sentence).split()


### Mô hình hóa

- Chuyển đoạn văn tiếng Việt về vector
- với các tham số là danh sách stopwords tiếng Việt
- và tokenizer tách từ tiếng Việt

In [ ]:
from sklearn.metrics import classification_report,accuracy_score
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
X = data_df['Compound_Content_SW']
y = data_df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25,
                                                   random_state=30)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Vector

In [ ]:
model_vector_CV = os.path.join("../../Vector/Vietnamese/CV", "Vietnamese_vectorizer_CV.joblib")
model_RFC_CV = os.path.join("../../Vector/Vietnamese/CV", "Vietnamese_RFC_model_CV.joblib")


In [ ]:
model_vector_TF = os.path.join("../../Vector/Vietnamese/TF", "Vietnamese_vectorizer_TF.joblib")
model_RFC_TF = os.path.join("../../Vector/Vietnamese/TF", "Vietnamese_RFC_model_TF.joblib")

In [ ]:
model_vector_W2V = os.path.join("../../Vector/Vietnamese/W2V", "Vietnamese_vectorizer_W2V.joblib")
model_RFC_W2V = os.path.join("../../Vector/Vietnamese/W2V", "Vietnamese_RFC_model_W2V.joblib")

In [ ]:
model_vector_D2V = os.path.join("../../Vector/Vietnamese/D2V", "Vietnamese_vectorizer_D2V.joblib")
model_RFC_D2V = os.path.join("../../Vector/Vietnamese/D2V", "Vietnamese_RFC_model_D2V.joblib")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec
import numpy as np

from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

## CountVectorizer

In [ ]:
vectorizerCV = CountVectorizer(
    stop_words = stopwords,
    tokenizer = tokenize,
)

In [ ]:
Xv_trainCV = vectorizerCV.fit_transform(X_train)
Xv_testCV = vectorizerCV.transform(X_test)

## TfidfVectorizer

In [ ]:
vectorizerTF = TfidfVectorizer(
    stop_words=stopwords,
    tokenizer = tokenize
    )

In [ ]:
Xv_trainTF = vectorizerTF.fit_transform(X_train)
Xv_testTF = vectorizerTF.transform(X_test)

## Word2Vec

In [ ]:
def makeWords(sentences):
    wordList = []
    for sentence in sentences:
        words = sentence.split(' ')
        wordList.append(words)
    return wordList

words_train_W2V = makeWords(X_train)
words_test_W2V = makeWords(X_test)

In [ ]:
# Train the Word2Vec model on the tokenized training sentences
vectorizerW2V = Word2Vec(
    sentences=words_train_W2V,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,
    epochs=20
)

In [ ]:
# Function to calculate the average Word2Vec vector for each sentence
def sentence_vector(sentence, model):
    # Get vectors for words in the sentence if they exist in the vocabulary
    vectors = [model.wv[word] for word in sentence if word in model.wv]
    if len(vectors) > 0:
        # Calculate the mean of the word vectors for the sentence
        return np.mean(vectors, axis=0)
    else:
        # Return a zero vector if no words in the sentence are in the vocabulary
        return np.zeros(model.vector_size)

In [ ]:
Xv_trainW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_train_W2V])
Xv_testW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_test_W2V])

## Doc2Vec

In [ ]:
documents = [TaggedDocument(doc.split(), [i]) for i, doc in enumerate(X_train)]

In [ ]:
vectorizerD2V = Doc2Vec(
    documents=documents,  # Truyền documents trực tiếp
    vector_size=100,      # Kích thước vector
    window=5,             # Kích thước cửa sổ ngữ cảnh
    min_count=1,          # Bỏ qua từ xuất hiện ít hơn min_count
    workers=4,            # Số luồng để huấn luyện
    epochs=20             # Số lần lặp để huấn luyện
)

In [ ]:
Xv_trainD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_train]
Xv_testD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_test]

## Save model

In [ ]:
# joblib.dump(vectorizerCV, model_vector_CV)
# joblib.dump(vectorizerTF, model_vector_TF)
# joblib.dump(vectorizerW2V,model_vector_W2V)
# joblib.dump(vectorizerD2V, model_vector_D2V)

# Train RandomForestClassifier

In [ ]:
rfc_cv = RandomForestClassifier()
rfc_tf = RandomForestClassifier()
rfc_w2v = RandomForestClassifier()
rfc_d2v = RandomForestClassifier()

## CountVectorizer

In [ ]:
rfc_cv.fit(Xv_trainCV,y_train)
pred_rfc_cv = rfc_cv.predict(Xv_testCV)

## TfidfVectorizer

In [ ]:
rfc_tf.fit(Xv_trainTF,y_train)
pred_rfc_tf = rfc_tf.predict(Xv_testTF)

## Word2Vec

In [ ]:
rfc_w2v.fit(Xv_trainW2V, y_train)
pred_rfc_w2v = rfc_w2v.predict(Xv_testW2V)

## Doc2Vec

In [ ]:
rfc_d2v.fit(Xv_trainD2V, y_train)
pred_rfc_d2v = rfc_d2v.predict(Xv_testD2V)

## Predict model

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_cv) 
print(accuracy)
print('-------------------\n')
print(classification_report(y_true=y_test, y_pred=pred_rfc_cv))

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_tf))

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_w2v))

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_d2v))

# Save model RFC

In [ ]:
# joblib.dump(rfc_cv, model_RFC_CV)
# joblib.dump(rfc_tf, model_RFC_TF)
# joblib.dump(vectorizerW2V,model_vector_W2V)
# joblib.dump(rfc_d2v, model_RFC_D2V)

# LSTM

## word2Vec

In [ ]:
# Reshape dữ liệu thành (samples, timesteps, features)
X_train_lstm = Xv_trainW2V.reshape(Xv_trainW2V.shape[0], 1, Xv_trainW2V.shape[1])
X_test_lstm = Xv_testW2V.reshape(Xv_testW2V.shape[0], 1, Xv_testW2V.shape[1])


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Xây dựng mô hình LSTM
model = Sequential([
    LSTM(128, input_shape=(1, 100), return_sequences=False),  # 100 là kích thước vector Word2Vec
    Dropout(0.3),  # Thêm Dropout để giảm overfitting
    Dense(64, activation='relu'),  # Lớp ẩn với hàm kích hoạt ReLU
    Dropout(0.3),  # Dropout lần nữa
    Dense(1, activation='sigmoid')  # Lớp đầu ra nhị phân
])

# Biên dịch mô hình
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Hiển thị cấu trúc mô hình
model.summary()


In [ ]:
# Huấn luyện mô hình
history = model.fit(
    X_train_lstm, y_train, 
    epochs=10,  # Số lần lặp
    batch_size=64,  # Kích thước batch
    validation_data=(X_test_lstm, y_test)  # Dữ liệu kiểm tra
)


In [ ]:

# Dự đoán
y_pred = (model.predict(X_test_lstm) > 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred) 
print(accuracy)
print('-------------------\n')
# Báo cáo kết quả
print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt

# Biểu đồ Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

# Biểu đồ Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()
plt.show()


## doc2vec

In [ ]:
import numpy as np

# Chuyển đổi thành numpy array và reshape
X_train_lstm = np.array(Xv_trainD2V).reshape(len(Xv_trainD2V), 1, 100)  # 1 là timestep
X_test_lstm = np.array(Xv_testD2V).reshape(len(Xv_testD2V), 1, 100)


In [ ]:
# Xây dựng mô hình
model = Sequential([
    LSTM(128, input_shape=(1, 100), return_sequences=False),  # 100 là kích thước vector
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Phân loại nhị phân
])

# Biên dịch mô hình
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Hiển thị cấu trúc mô hình
model.summary()


In [ ]:
# Huấn luyện mô hình
history = model.fit(
    X_train_lstm, y_train,
    epochs=10,              # Số lần lặp
    batch_size=64,          # Kích thước batch
    validation_data=(X_test_lstm, y_test)  # Dữ liệu kiểm tra
)


In [ ]:
# Dự đoán
y_pred = (model.predict(X_test_lstm) > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred) 
print(accuracy)
print('-------------------\n')
# Báo cáo kết quả
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt

# Vẽ Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

# Vẽ Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()
plt.show()
